$$
MQI = \sum_{t=1}^{T-1} \sqrt{(x_{t+1} - x_t)^2 + (y_{t+1} - y_t)^2 + (z_{t+1} - z_t)^2}
$$

$$
MQI = \sum_{t} \sqrt{(x_{t+1} - x_t)^2 + (y_{t+1} - y_t)^2 + (z_{t+1} - z_t)^2}
$$

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.signal import butter, filtfilt

# --------------------------
# 0) Chemins du projet
# --------------------------
CODE_DIR = Path().resolve()              # .../SYNCOGEST/Code/Codes MP
PROJECT_ROOT = CODE_DIR.parent.parent    # .../SYNCOGEST
DATA_DIR = PROJECT_ROOT / "DATA"

VIDEO_DIR = DATA_DIR / "Video"
EXCEL_DIR = DATA_DIR / "Excels_code"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("VIDEO_DIR =", VIDEO_DIR)
print("EXCEL_DIR =", EXCEL_DIR)
# ========= PARAMS =========

ROOT = VIDEO_DIR  # adapte
POSE_FILES = list(ROOT.rglob("*_pose.xlsx"))

ORDER = 4
CUTOFF_HZ = 6.0  # 5–6 Hz typique pour gestes / tête
DEFAULT_FS = 30.0  # si on ne peut pas estimer la fréquence depuis t_ms

# ========= UTILS =========
def butter_lowpass_filt(x, fs, cutoff=CUTOFF_HZ, order=ORDER):
    x = np.asarray(x, dtype=float)
    if len(x) < (order * 3 + 1):
        return x  # trop court -> pas de filtre
    nyq = 0.5 * fs
    w = cutoff / nyq
    w = min(w, 0.99)  # sécurité
    b, a = butter(order, w, btype="low", analog=False)
    # filtfilt = zero-phase (pas de décalage temporel)
    return filtfilt(b, a, x, method="pad")

def estimate_fs(df):
    # si ton pose.xlsx contient t_ms, on estime la fréquence
    if "t_ms" in df.columns:
        t = df["t_ms"].to_numpy(dtype=float)
        dt = np.diff(t) / 1000.0
        dt = dt[np.isfinite(dt)]
        if len(dt) > 10:
            med = np.median(dt)
            if med > 0:
                return 1.0 / med
    return DEFAULT_FS

def qdm_2d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    dx = np.diff(x)
    dy = np.diff(y)
    step = np.sqrt(dx*dx + dy*dy)
    step = step[np.isfinite(step)]
    return float(step.sum()), int(step.size)

def shoulder_width_2d(ls_x, ls_y, rs_x, rs_y):
    sw = np.sqrt((ls_x - rs_x)**2 + (ls_y - rs_y)**2)
    sw = sw[np.isfinite(sw)]
    if len(sw) == 0:
        return np.nan, np.nan
    return float(np.median(sw)), float(np.mean(sw))

# ========= MAIN LOOP =========
rows = []

for f in POSE_FILES:
    df = pd.read_excel(f)

    # colonnes nécessaires (MediaPipe PoseLandmark)
    needed = [
        "LEFT_WRIST_x","LEFT_WRIST_y",
        "RIGHT_WRIST_x","RIGHT_WRIST_y",
        "NOSE_x","NOSE_y",
        "LEFT_SHOULDER_x","LEFT_SHOULDER_y",
        "RIGHT_SHOULDER_x","RIGHT_SHOULDER_y",
    ]
    if not all(c in df.columns for c in needed):
        # Skip si export incomplet
        continue

    fs = estimate_fs(df)

    # --- Filtrage Butterworth sur les trajectoires (2D) ---
    LW_x = butter_lowpass_filt(df["LEFT_WRIST_x"], fs)
    LW_y = butter_lowpass_filt(df["LEFT_WRIST_y"], fs)
    RW_x = butter_lowpass_filt(df["RIGHT_WRIST_x"], fs)
    RW_y = butter_lowpass_filt(df["RIGHT_WRIST_y"], fs)

    NO_x = butter_lowpass_filt(df["NOSE_x"], fs)
    NO_y = butter_lowpass_filt(df["NOSE_y"], fs)

    LS_x = butter_lowpass_filt(df["LEFT_SHOULDER_x"], fs)
    LS_y = butter_lowpass_filt(df["LEFT_SHOULDER_y"], fs)
    RS_x = butter_lowpass_filt(df["RIGHT_SHOULDER_x"], fs)
    RS_y = butter_lowpass_filt(df["RIGHT_SHOULDER_y"], fs)

    # --- QDM (déplacement cumulé 2D) ---
    q_lw, n_lw = qdm_2d(LW_x, LW_y)
    q_rw, n_rw = qdm_2d(RW_x, RW_y)
    q_wrists = q_lw + q_rw

    q_nose, n_nose = qdm_2d(NO_x, NO_y)

    # --- Shoulder width MP (apparent, unités normalisées) ---
    sw_med, sw_mean = shoulder_width_2d(LS_x, LS_y, RS_x, RS_y)

    # --- Identifiants ---
    video_id = f.stem.replace("_pose", "")  # ex: SEATEDD01_P1 ou SEATEDD01 selon ton naming
    rel = f.relative_to(ROOT).parts
    D = rel[0] if len(rel) > 0 else ""
    P = rel[1] if len(rel) > 1 else ""
    condition = rel[2] if len(rel) > 2 else ""

    rows.append({
        "video_id": video_id,
        "D": D,
        "P": P,
        "condition": condition,
        "fs_est": fs,

        "SW_mp_median": sw_med,
        "SW_mp_mean": sw_mean,

        "QDM_LEFT_WRIST_mp_filt": q_lw,
        "QDM_RIGHT_WRIST_mp_filt": q_rw,
        "QDM_WRISTS_mp_filt": q_wrists,

        "QDM_NOSE_mp_filt": q_nose,

        "n_steps_wrist_L": n_lw,
        "n_steps_wrist_R": n_rw,
        "n_steps_nose": n_nose,

        # normalisation “sans unité” (en largeurs d'épaules MP)
        "QDM_WRISTS_mp_norm": (q_wrists / sw_med) if np.isfinite(sw_med) and sw_med > 0 else np.nan,
        "QDM_NOSE_mp_norm": (q_nose / sw_med) if np.isfinite(sw_med) and sw_med > 0 else np.nan,
    })

df_mp = pd.DataFrame(rows).sort_values(["D","P","condition","video_id"])
out1 = EXCEL_DIR / "mediapipe_QDM_filtered_and_SW.xlsx"
df_mp.to_excel(out1, index=False)
print("✅ Saved:", out1)

PROJECT_ROOT = /Users/matysprecloux/Desktop/SYNCOGEST
VIDEO_DIR = /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video
EXCEL_DIR = /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Excels_code
✅ Saved: /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Excels_code/mediapipe_QDM_filtered_and_SW.xlsx


In [2]:
import pandas as pd
from pathlib import Path


# --------------------------
# 0) Chemins du projet
# --------------------------
CODE_DIR = Path().resolve()              # .../SYNCOGEST/Code/Codes MP
PROJECT_ROOT = CODE_DIR.parent.parent    # .../SYNCOGEST
DATA_DIR = PROJECT_ROOT / "DATA"

VIDEO_DIR = DATA_DIR / "Video"
EXCEL_DIR = DATA_DIR / "Excels_code"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("VIDEO_DIR =", VIDEO_DIR)
print("EXCEL_DIR =", EXCEL_DIR)

ROOT = EXCEL_DIR

mp_path = ROOT / "mediapipe_QDM_filtered_and_SW.xlsx"
vicon_sw_path = ROOT / "vicon_shoulder_width_by_csv.xlsx"  # celui avec P1/P2_shoulder_width_median_mm

mp = pd.read_excel(mp_path)
vsw = pd.read_excel(vicon_sw_path)

# --- clé vidéo côté Vicon (nom du fichier csv) ---
# vsw["csv"] = chemin vers ".../SEATEDD01.csv"
vsw["video_base"] = vsw["csv"].astype(str).apply(lambda x: Path(x).stem)  # "SEATEDD01"

# --- extraire base + participant depuis video_id MP ---
# ex "SEATEDD01_P1" -> base="SEATEDD01", pid="P1"
tmp = mp["video_id"].astype(str).str.rsplit("_", n=1, expand=True)
mp["video_base"] = tmp[0]
mp["pid"] = tmp[1]  # "P1" ou "P2"

# --- récupérer SW vicon mm selon pid ---
# on crée 2 colonnes dans vsw : SW_vicon_P1_mm et SW_vicon_P2_mm
vsw = vsw.rename(columns={
    "P1_shoulder_width_median_mm": "SW_vicon_P1_mm",
    "P2_shoulder_width_median_mm": "SW_vicon_P2_mm"
})
mp_set = set(mp["video_base"])
vsw_set = set(vsw["video_base"])

print("Dans MP mais pas Vicon :")
for x in sorted(mp_set - vsw_set):
    print("-", repr(x))

print("\nDans Vicon mais pas MP :")
for x in sorted(vsw_set - mp_set):
    print("-", repr(x))
    
merged = mp.merge(
    vsw[["video_base", "SW_vicon_P1_mm", "SW_vicon_P2_mm"]],
    on="video_base",
    how="left"
)

# choisir la bonne largeur selon pid
merged["SW_vicon_mm"] = merged.apply(
    lambda r: r["SW_vicon_P1_mm"] if r["pid"] == "P1" else r["SW_vicon_P2_mm"],
    axis=1
)

# scale
merged["scale_vicon_over_mp"] = merged["SW_vicon_mm"] / merged["SW_mp_median"]

# MediaPipe -> mm estimés (après filtre)
merged["QDM_WRISTS_mp_mm_est"] = merged["QDM_WRISTS_mp_norm"] * merged["SW_vicon_mm"]
merged["QDM_NOSE_mp_mm_est"] = merged["QDM_NOSE_mp_norm"] * merged["SW_vicon_mm"]

# sauvegarde
out2 = EXCEL_DIR / "mediapipe_QDM_filtered_scaled_to_mm.xlsx"
merged.to_excel(out2, index=False)
print("✅ Saved:", out2)

# contrôle rapide
print("Missing SW_vicon_mm:", merged["SW_vicon_mm"].isna().sum(), "/", len(merged))
print("Missing SW_mp_median:", merged["SW_mp_median"].isna().sum(), "/", len(merged))

PROJECT_ROOT = /Users/matysprecloux/Desktop/SYNCOGEST
VIDEO_DIR = /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video
EXCEL_DIR = /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Excels_code
Dans MP mais pas Vicon :

Dans Vicon mais pas MP :
✅ Saved: /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Excels_code/mediapipe_QDM_filtered_scaled_to_mm.xlsx
Missing SW_vicon_mm: 0 / 120
Missing SW_mp_median: 0 / 120


In [5]:
import pandas as pd
import re
from pathlib import Path


# --- charge ton fichier Vicon QDM ---
df = pd.read_excel(EXCEL_DIR / "vicon_QDM_wrists_head_normByShoulders.xlsx")

# =========================
# 1) clé vidéo canonique
# =========================
def normalize_video_base(s):
    s = str(s).upper()
    s = s.replace("_", "").replace("-", "").replace(" ", "")
    s = s.replace("SEMI", "SEMID")
    m = re.search(r"(SEATED|SEMID|STANDING)D\d+", s)
    return m.group(0) if m else s

df["video_base_norm"] = df["video_id"].apply(normalize_video_base)

# =========================
# 2) passer de large → long
# =========================
rows = []

for _, r in df.iterrows():
    for pid in ["P1", "P2"]:
        rows.append({
            "video_base_norm": r["video_base_norm"],
            "pid": pid,

            "QDM_WRISTS_vicon_mm": r[f"{pid}_QDM_WRISTS_mm"],
            "QDM_HEAD_vicon_mm":   r[f"{pid}_QDM_HEAD_mm"],

            # optionnel mais utile
            "condition": r["video_base_norm"].split("D")[0],
        })

df_long = pd.DataFrame(rows)

# =========================
# 3) sauvegarde
# =========================
out = EXCEL_DIR / "vicon_QDM_for_validation_long.xlsx"
df_long.to_excel(out, index=False)
print("✅ Saved:", out)

✅ Saved: /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Excels_code/vicon_QDM_for_validation_long.xlsx


In [6]:
import pandas as pd
import numpy as np
import re

# ---------
# 1) Charger le fichier final
# ---------
df = pd.read_excel(EXCEL_DIR / "FINAL_FILE_FOR_JASP.xlsx")

# ---------
# 2) Passer du format large au format long
# ---------
rows = []

for _, r in df.iterrows():
    # P1
    rows.append({
        "video_base_norm": r["video_base_norm"],
        "condition": r["condition"],
        "pid": "P1",
        "WRIST_mp_mm": r["P1_WRIST_MEDIAPIPE_norm_mm"],
        "WRIST_VICON_mm": r["P1_WRIST_VICON_norm_mm"],
        "HEAD_mp_mm": r["P1_HEAD_MEDIAPIPE_norm_mm"],
        "HEAD_VICON_mm": r["P1_HEAD_VICON_norm_mm"],
    })

    # P2
    rows.append({
        "video_base_norm": r["video_base_norm"],
        "condition": r["condition"],
        "pid": "P2",
        "WRIST_mp_mm": r["P2_WRIST_MEDIAPIPE_norm_mm"],
        "WRIST_VICON_mm": r["P2_WRIST_VICON_norm_mm"],
        "HEAD_mp_mm": r["P2_HEAD_MEDIAPIPE_norm_mm"],
        "HEAD_VICON_mm": r["P2_HEAD_VICON_norm_mm"],
    })

df_long = pd.DataFrame(rows)

# ---------
# 3) Extraire la dyade Dxx
# ---------
def get_dyad(s):
    m = re.search(r"(D\d+)", str(s).upper())
    return m.group(1) if m else "D??"

df_long["dyad"] = df_long["video_base_norm"].apply(get_dyad)

# ---------
# 4) Choisir les variables à corréler
# ---------
XCOL = "WRIST_mp_mm"
YCOL = "WRIST_VICON_mm"
COND = "condition"
VID  = "video_base_norm"
PID  = "pid"

# sécurité
df_long = df_long.dropna(subset=[XCOL, YCOL, COND, "dyad", PID]).copy()

# ---------
# 5) Corrélation observée
# ---------
x = df_long[XCOL].to_numpy(float)
y = df_long[YCOL].to_numpy(float)
r_obs = np.corrcoef(x, y)[0, 1]

# ---------
# 6) Permutation robuste : swap P1/P2 au sein de chaque (dyad, condition)
# ---------
rng = np.random.default_rng(42)
n_perm = 5000
r_perm = np.empty(n_perm)

groups = df_long.groupby(["dyad", COND], sort=False)

bad = []
for key, g in groups:
    if len(g) != 2:
        bad.append((key, len(g)))

if bad:
    print("⚠️ Groupes non conformes (pas 2 lignes):", bad[:10])

for i in range(n_perm):
    y_shuf = df_long[YCOL].to_numpy(float).copy()

    for (dyad, cond), g in groups:
        idx = g.index.to_list()
        if len(idx) != 2:
            continue
        if rng.random() < 0.5:
            y_shuf[idx[0]], y_shuf[idx[1]] = y_shuf[idx[1]], y_shuf[idx[0]]

    r_perm[i] = np.corrcoef(df_long[XCOL].to_numpy(float), y_shuf)[0, 1]

p_perm = (np.sum(np.abs(r_perm) >= abs(r_obs)) + 1) / (n_perm + 1)

print("r_obs =", r_obs)
print("Permutation (swap within dyad×condition): p =", p_perm)
print("Null mean r =", r_perm.mean(), "sd =", r_perm.std())

r_obs = 0.926968148108207
Permutation (swap within dyad×condition): p = 0.0001999600079984003
Null mean r = 0.6793193791726544 sd = 0.10168484717883758
